# 🤖 Phase 4.5 — LLM Integration Demo

> **Gemma 4 E4B Threat Intelligence via Ollama**

This notebook demonstrates the LLM-powered threat analysis pipeline:
1. Ollama health check & model availability
2. RAM-based model selection
3. Threat analysis on malicious feature sets
4. Safety confirmation on benign feature sets
5. ThreatReport generation (Markdown + JSON)
6. LLM inference benchmarking
7. Interactive follow-up Q&A
8. Prompt strategy comparison

In [ ]:
import sys, time, json
sys.path.insert(0, '..')

from src.llm.client import GemmaClient
from src.llm.analyzer import ThreatAnalyzer
from src.llm.report_generator import ThreatReport, generate_file_hash
from src.llm.prompts import (
    SYSTEM_PROMPT, THREAT_ANALYSIS_TEMPLATE,
    format_feature_summary, format_suspicious_features,
    FEATURE_DESCRIPTIONS
)
from src.config import FEATURE_COLUMNS

print('All LLM modules imported successfully')
print(f'Feature columns: {len(FEATURE_COLUMNS)}')
print(f'System prompt: {len(SYSTEM_PROMPT)} chars')
print(f'Feature descriptions: {len(FEATURE_DESCRIPTIONS)} entries')

## Cell 1: Ollama Health & Model Availability

In [ ]:
client = GemmaClient()
print(f'Client: {client}')
print()

# Health check
healthy = client.check_health()
print(f'Ollama reachable: {healthy}')

if not healthy:
    print('\n--- Ollama is NOT running ---')
    print('To start Ollama:')
    print('  1. Install: winget install Ollama.Ollama')
    print('  2. Pull model: ollama pull gemma4:e4b')
    print('  3. Start: ollama serve')
    print('\nThe remaining cells will demonstrate offline capabilities.')

## Cell 2: RAM Check & Model Selection

In [ ]:
import psutil

mem = psutil.virtual_memory()
print(f'Total RAM: {mem.total / (1024**3):.1f} GB')
print(f'Available: {mem.available / (1024**3):.1f} GB ({100-mem.percent:.1f}% free)')
print(f'Used: {mem.used / (1024**3):.1f} GB ({mem.percent}%)')
print()

selected = client.check_ram()
print(f'\nSelected model: {selected}')
print(f'Client status: {client}')

# Show selection logic
avail_mb = mem.available / (1024*1024)
print(f'\nSelection logic:')
print(f'  >= 5120 MB -> gemma4:e4b (primary)  {"<-- SELECTED" if avail_mb >= 5120 else ""}')
print(f'  >= 3072 MB -> gemma4:e2b (fallback) {"<-- SELECTED" if 3072 <= avail_mb < 5120 else ""}')
print(f'  <  3072 MB -> Disabled              {"<-- CURRENT" if avail_mb < 3072 else ""}')

## Cell 3: Threat Analysis on Malicious Feature Sets

In [ ]:
analyzer = ThreatAnalyzer(client)

# 3 malicious feature sets
malicious_samples = [
    {
        'name': 'JS Exploit PDF',
        'features': {col: 0.0 for col in FEATURE_COLUMNS},
        'overrides': {'js_count': 5, 'javascript_count': 3, 'openaction_count': 1,
                      'action_count': 2, 'obfuscation_count': 12, 'filter_count': 8,
                      'obj_count': 45, 'stream_count': 15, 'pdf_size': 85000, 'page_count': 1},
        'confidence': 0.973,
    },
    {
        'name': 'Embedded Payload PDF',
        'features': {col: 0.0 for col in FEATURE_COLUMNS},
        'overrides': {'embedded_file_count': 3, 'launch_count': 1, 'objstm_count': 5,
                      'filter_count': 12, 'obfuscation_count': 8, 'obj_count': 120,
                      'stream_count': 40, 'pdf_size': 250000, 'page_count': 1, 'has_text': 0},
        'confidence': 0.891,
    },
    {
        'name': 'URI Redirect PDF',
        'features': {col: 0.0 for col in FEATURE_COLUMNS},
        'overrides': {'uri_count': 8, 'submitform_count': 2, 'acroform_count': 3,
                      'action_count': 5, 'xfa_count': 1, 'obj_count': 30,
                      'pdf_size': 45000, 'page_count': 2},
        'confidence': 0.845,
    },
]

for sample in malicious_samples:
    features = {**sample['features'], **sample['overrides']}
    print(f"\n{'='*60}")
    print(f"  {sample['name']}")
    print(f"{'='*60}")
    
    suspicious = analyzer.identify_suspicious_features(features)
    print(f'Suspicious features: {len(suspicious)}')
    for name, val, dev, desc in suspicious[:5]:
        emoji = 'HIGH' if abs(dev) > 5 else 'MED' if abs(dev) > 3 else 'LOW'
        print(f'  [{emoji}] {name} = {val:.0f} ({dev:+.1f} sigma)')
    
    severity = ThreatAnalyzer._extract_severity('', 'Malicious', suspicious)
    print(f'Inferred severity: {severity}')
    
    # If Ollama available, run full analysis
    if client.is_available and client.model:
        report = analyzer.analyze(features, 'Malicious', sample['confidence'],
                                  filename=f"{sample['name'].replace(' ', '_')}.pdf")
        print(f'\nLLM Analysis ({report.processing_time_ms:.0f}ms):')
        print(report.threat_explanation[:500])
    else:
        print('\n[Ollama offline - showing feature analysis only]')

## Cell 4: Benign Feature Set Analysis

In [ ]:
benign_samples = [
    {
        'name': 'Normal Document',
        'features': {col: 0.0 for col in FEATURE_COLUMNS},
        'overrides': {'obj_count': 15, 'endobj_count': 15, 'stream_count': 5,
                      'endstream_count': 5, 'xref_count': 1, 'trailer_count': 1,
                      'startxref_count': 1, 'pdf_size': 120000, 'page_count': 5,
                      'has_text': 1, 'header_valid': 1, 'font_obj_count': 3, 'image_count': 2},
        'confidence': 0.95,
    },
    {
        'name': 'Simple Invoice',
        'features': {col: 0.0 for col in FEATURE_COLUMNS},
        'overrides': {'obj_count': 8, 'endobj_count': 8, 'stream_count': 2,
                      'endstream_count': 2, 'xref_count': 1, 'trailer_count': 1,
                      'startxref_count': 1, 'pdf_size': 35000, 'page_count': 1,
                      'has_text': 1, 'header_valid': 1, 'font_obj_count': 2},
        'confidence': 0.98,
    },
]

for sample in benign_samples:
    features = {**sample['features'], **sample['overrides']}
    print(f"\n{'='*60}")
    print(f"  {sample['name']} (Benign)")
    print(f"{'='*60}")
    
    suspicious = analyzer.identify_suspicious_features(features)
    print(f'Suspicious features: {len(suspicious)} (expected: 0 or very few)')
    
    if client.is_available and client.model:
        summary = analyzer.quick_summary(features, 'Benign', sample['confidence'])
        print(f'\nQuick Summary:\n{summary[:300]}')
    else:
        print('[Ollama offline - feature analysis only]')

## Cell 5: ThreatReport Formatting (Markdown + JSON)

In [ ]:
# Create a demo report
demo_report = ThreatReport(
    file_hash=generate_file_hash(b'demo malicious pdf'),
    filename='suspicious_invoice.pdf',
    ml_prediction='Malicious',
    ml_confidence=0.973,
    risk_severity='Critical',
    threat_explanation=(
        'This PDF exhibits a classic JavaScript exploit chain. The presence of '
        '5 /JS tags combined with /OpenAction indicates auto-execution of malicious '
        'code upon document open. High obfuscation count (12) suggests hex-encoding '
        'to evade signature-based detection. The attack pattern maps to MITRE '
        'ATT&CK T1059.007 (JavaScript Execution) and T1204.002 (Malicious File).'
    ),
    attack_vector='JavaScript Exploit: /OpenAction -> /JS -> Heap Spray',
    suspicious_features=[
        ('js_count', 5.0, 8.2, '/JS tags'),
        ('openaction_count', 1.0, 4.5, '/OpenAction auto-execute'),
        ('obfuscation_count', 12.0, 6.1, 'Hex-encoded obfuscation'),
    ],
    remediation='1. Quarantine immediately\n2. Do not open\n3. Block SHA-256 hash',
    processing_time_ms=42.5,
    model_used='gemma4:e4b',
    feature_summary={'js_count': 5.0, 'openaction_count': 1.0, 'obfuscation_count': 12.0},
)

# Markdown
md = demo_report.to_markdown()
print('=== MARKDOWN REPORT ===')
print(md[:800])
print(f'\n... ({len(md)} total chars)')

# JSON
js = demo_report.to_json()
print(f'\n=== JSON REPORT ({len(js)} chars) ===')
print(js[:500])

# Roundtrip
restored = ThreatReport.from_json(js)
assert restored.file_hash == demo_report.file_hash
print(f'\nJSON roundtrip: PASS')

## Cell 6: LLM Inference Benchmark

In [ ]:
import numpy as np

if client.is_available and client.model:
    print(f'Benchmarking LLM inference ({client.model})...')
    print('Running 5 iterations...')
    
    test_prompt = 'In 2 sentences, what makes /JS in a PDF suspicious?'
    times = []
    
    # Warmup
    client.warmup()
    
    for i in range(5):
        start = time.perf_counter()
        resp = client.generate(test_prompt, system_prompt=SYSTEM_PROMPT)
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)
        print(f'  Run {i+1}: {elapsed:.0f}ms ({len(resp)} chars)')
    
    times_arr = np.array(times)
    print(f'\nResults:')
    print(f'  Mean: {times_arr.mean():.0f}ms')
    print(f'  Std:  {times_arr.std():.0f}ms')
    print(f'  Min:  {times_arr.min():.0f}ms')
    print(f'  Max:  {times_arr.max():.0f}ms')
    print(f'  Target: <30,000ms (NFR-103): {"PASS" if times_arr.mean() < 30000 else "FAIL"}')
else:
    print('Ollama not available - skipping benchmark')
    print('To benchmark, start Ollama and re-run this cell')

## Cell 7: Interactive Follow-Up Q&A

In [ ]:
if client.is_available and client.model:
    # Use the demo report for context
    questions = [
        'What specific CVE could this exploit target?',
        'How can I detect similar PDFs in bulk?',
        'Is this attack pattern common in targeted campaigns?',
    ]
    
    for q in questions:
        print(f'\nQ: {q}')
        answer = analyzer.follow_up(q, demo_report)
        print(f'A: {answer[:300]}...')
        print('-' * 40)
else:
    print('Follow-up Q&A requires Ollama to be running.')
    print('\nExample questions that would be asked:')
    print('  1. What specific CVE could this exploit target?')
    print('  2. How can I detect similar PDFs in bulk?')
    print('  3. Is this attack pattern common in targeted campaigns?')

## Cell 8: Prompt Strategy Comparison

In [ ]:
# Compare system prompt characteristics
print('=== Prompt Strategy Analysis ===')
print(f'\nSystem prompt length: {len(SYSTEM_PROMPT)} chars')
print(f'System prompt sections:')
for line in SYSTEM_PROMPT.split('\n'):
    if line.startswith('##'):
        print(f'  - {line.strip("# ")}')

# Show template sizes
from src.llm.prompts import (
    THREAT_ANALYSIS_TEMPLATE, JAVASCRIPT_ANALYSIS_TEMPLATE,
    QUICK_SUMMARY_TEMPLATE, FOLLOW_UP_TEMPLATE
)

templates = {
    'Threat Analysis': THREAT_ANALYSIS_TEMPLATE,
    'JavaScript Analysis': JAVASCRIPT_ANALYSIS_TEMPLATE,
    'Quick Summary': QUICK_SUMMARY_TEMPLATE,
    'Follow-Up': FOLLOW_UP_TEMPLATE,
}

print(f'\nTemplate sizes:')
for name, tmpl in templates.items():
    placeholders = [p for p in tmpl.split('{') if '}' in p]
    print(f'  {name}: {len(tmpl)} chars, {len(placeholders)} placeholders')

print(f'\nDesign: Zero-shot with structured system prompt')
print('Rationale: Gemma 4 E4B performs best with detailed instructions')
print('           rather than few-shot examples, which consume context window')

## Summary

| Component | Status | Details |
|-----------|--------|---------|
| `GemmaClient` | Implemented | Health check, RAM selection, sync/stream generation |
| `prompts.py` | Implemented | 5 templates, SOC analyst persona, MITRE ATT&CK |
| `ThreatAnalyzer` | Implemented | Feature analysis, LLM orchestration, caching |
| `ThreatReport` | Implemented | Markdown/JSON/dict export, roundtrip serialization |
| Ollama Integration | Ready | Auto-detect, fallback, graceful degradation |